In [1]:
%pip install truststore

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install deep_translator

Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install langdetect

In [ ]:
%pip install truststore streamlit transformers torch
%pip install deep-translator langdetect gTTS
%pip install audio-recorder-streamlit SpeechRecognition
%pip install librosa soundfile

In [ ]:
%pip install --upgrade "transformers<5"

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
import truststore
truststore.inject_into_ssl()
import streamlit as st
from transformers import pipeline
from deep_translator import GoogleTranslator
from langdetect import detect
from gtts import gTTS
from audio_recorder_streamlit import audio_recorder
import speech_recognition as sr
import tempfile
import os

# ----------------------------------
# Page Config
# ----------------------------------

st.set_page_config(
    page_title="Multilingual AI Voice Assistant",
    page_icon="🌍",
    layout="wide"
)

# ----------------------------------
# Hugging Face Model
# ----------------------------------

@st.cache_resource
def load_model():

    model = pipeline(
        "text2text-generation",
        model="google/flan-t5-base"
    )

    return model

chatbot = load_model()

# ----------------------------------
# Speech To Text
# ----------------------------------

def speech_to_text(audio_file):

    recognizer = sr.Recognizer()

    try:

        with sr.AudioFile(audio_file) as source:

            audio_data = recognizer.record(source)

            text = recognizer.recognize_google(
                audio_data
            )

            return text

    except Exception as e:

        return f"Error: {e}"

# ----------------------------------
# AI Response
# ----------------------------------

def generate_response(text):

    result = chatbot(
        text,
        max_new_tokens=100
    )

    return result[0]["generated_tex"]
# Text To Speech
# ----------------------------------

def text_to_speech(text, lang):

    tts = gTTS(
        text=text,
        lang=lang
    )

    file_path = "reply.mp3"

    tts.save(file_path)

    return file_path

# ----------------------------------
# UI
# ----------------------------------

st.title("🌍 Multilingual AI Voice Assistant")

st.markdown("""
Speak in:

English  
Tamil  
Hindi  

The assistant will understand,
translate,
generate AI response,
and speak back.
""")

# ----------------------------------
# Session State
# ----------------------------------

if "history" not in st.session_state:

    st.session_state.history = []

# ----------------------------------
# Voice Input
# ----------------------------------

st.subheader("🎤 Record Voice")

audio_bytes = audio_recorder(
    pause_threshold=2.0
)

if audio_bytes:

    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".wav"
    ) as temp_audio:

        temp_audio.write(audio_bytes)

        temp_path = temp_audio.name

    st.audio(audio_bytes)

    user_text = speech_to_text(temp_path)

    os.remove(temp_path)

    st.success(
        f"Recognized Text: {user_text}"
    )

    try:

        # Detect language

        detected_lang = detect(user_text)

        st.info(
            f"Detected Language: {detected_lang}"
        )

        # Translate To English

        english_text = GoogleTranslator(
            source='auto',
            target='en'
        ).translate(user_text)

        st.write(
            f"Translated: {english_text}"
        )

        # AI Response

        ai_response = generate_response(
            english_text
        )

        # Translate Back

        translated_response = GoogleTranslator(
            source='en',
            target=detected_lang
        ).translate(ai_response)

        st.subheader(
            "🤖 Assistant Response"
        )

        st.write(
            translated_response
        )

        # Store Chat

        st.session_state.history.append(
            (
                user_text,
                translated_response
            )
        )

        # Voice Output

        mp3_file = text_to_speech(
            translated_response,
            detected_lang
        )

        audio_file = open(
            mp3_file,
            "rb"
        )

        audio_bytes = audio_file.read()

        st.audio(
            audio_bytes,
            format="audio/mp3"
        )

        audio_file.close()

        os.remove(mp3_file)

    except Exception as e:

        st.error(e)

# ----------------------------------
# Text Chat Option
# ----------------------------------

st.subheader("💬 Text Chat")

user_input = st.text_input(
    "Ask anything"
)

if st.button("Send"):

    if user_input:

        detected_lang = detect(
            user_input
        )

        english_text = GoogleTranslator(
            source="auto",
            target="en"
        ).translate(
            user_input
        )

        ai_response = generate_response(
            english_text
        )

        translated_response = GoogleTranslator(
            source="en",
            target=detected_lang
        ).translate(
            ai_response
        )

        st.write(
            "🤖",
            translated_response
        )

        mp3_file = text_to_speech(
            translated_response,
            detected_lang
        )

        st.audio(
            mp3_file
        )

        st.session_state.history.append(
            (
                user_input,
                translated_response
            )
        )

# ----------------------------------
# Chat History
# ----------------------------------

st.subheader("📜 Conversation History")

for q, a in st.session_state.history[::-1]:

    st.markdown(
        f"""
        **👤 User:** {q}

        **🤖 Assistant:** {a}
        """
    )


In [ ]:
%pip install --upgrade --force-reinstall "transformers==4.46.3" "tokenizers>=0.20,<0.21" sentencepiece

In [ ]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)

from transformers import pipeline

whisper_model = pipeline(
    "automatic-speech-recognition",
    model="openai/whisper-small"
)

print("Whisper loaded successfully!")

In [ ]:
import truststore
truststore.inject_into_ssl()

import streamlit as st
from transformers import pipeline
from deep_translator import GoogleTranslator
from langdetect import detect
from gtts import gTTS
from audio_recorder_streamlit import audio_recorder
import tempfile
import os

# ----------------------------------
# Page Config
# ----------------------------------

st.set_page_config(
    page_title="Multilingual AI Voice Assistant",
    page_icon="🌍",
    layout="wide"
)

# ----------------------------------
# Hugging Face Whisper Model
# ----------------------------------

@st.cache_resource
def load_whisper_model():

    model = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small"
    )

    return model


whisper_model = load_whisper_model()

# ----------------------------------
# Hugging Face Chatbot Model
# ----------------------------------

@st.cache_resource
def load_chatbot():

    model = pipeline(
        "text2text-generation",
        model="google/flan-t5-base"
    )

    return model


chatbot = load_chatbot()

# ----------------------------------
# Speech To Text Using Whisper
# ----------------------------------

def speech_to_text(audio_file):

    try:
        result = whisper_model(
            audio_file,
            generate_kwargs={"task": "transcribe"}
        )

        return result["text"].strip()

    except Exception as e:
        return f"Error: {e}"

# ----------------------------------
# AI Response
# ----------------------------------

def generate_response(text):

    result = chatbot(
        text,
        max_new_tokens=100
    )

    return result[0]["generated_text"]

# ----------------------------------
# Voice Recording
# ----------------------------------

st.title("🌍 Multilingual AI Voice Assistant")

st.write("Click the microphone and speak.")

audio_bytes = audio_recorder()

if audio_bytes:

    with tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ) as temp_audio:

        temp_audio.write(audio_bytes)
        audio_path = temp_audio.name

    try:
        # Convert speech into text
        user_text = speech_to_text(audio_path)

        if user_text.startswith("Error:"):
            st.error(user_text)

        elif user_text:

            st.write("**Recognized Text:**", user_text)

            # Generate chatbot response
            response = generate_response(user_text)

            st.write("**AI Response:**", response)

    finally:
        if os.path.exists(audio_path):
            os.remove(audio_path)
```

In [ ]:
import truststore
truststore.inject_into_ssl()

import streamlit as st
from transformers import pipeline
from deep_translator import GoogleTranslator
from langdetect import detect
from gtts import gTTS
from audio_recorder_streamlit import audio_recorder
import tempfile
import os

# ----------------------------------
# Page Config
# ----------------------------------

st.set_page_config(
    page_title="Multilingual AI Voice Assistant",
    page_icon="🌍",
    layout="wide"
)

# ----------------------------------
# Load Whisper Model
# ----------------------------------

@st.cache_resource
def load_whisper_model():

    model = pipeline(
        "automatic-speech-recognition",
        model="openai/whisper-small"
    )

    return model


whisper_model = load_whisper_model()

# ----------------------------------
# Load Chatbot Model
# ----------------------------------

@st.cache_resource
def load_chatbot():

    model = pipeline(
        "text2text-generation",
        model="google/flan-t5-base"
    )

    return model


chatbot = load_chatbot()

# ----------------------------------
# Speech To Text Using Whisper
# ----------------------------------

def speech_to_text(audio_path):

    try:

        result = whisper_model(
            audio_path,
            generate_kwargs={
                "task": "transcribe"
            }
        )

        return result["text"].strip()

    except Exception as e:

        raise Exception(
            f"Whisper speech recognition failed: {e}"
        )

# ----------------------------------
# Generate AI Response
# ----------------------------------

def generate_response(text):

    result = chatbot(
        text,
        max_new_tokens=100
    )

    return result[0]["generated_text"]

# ----------------------------------
# Translate Text
# ----------------------------------

def translate_text(text, source, target):

    if source == target:
        return text

    return GoogleTranslator(
        source=source,
        target=target
    ).translate(text)

# ----------------------------------
# Text To Speech
# ----------------------------------

def text_to_speech(text, lang):

    with tempfile.NamedTemporaryFile(
        delete=False,
        suffix=".mp3"
    ) as temp_audio:

        audio_path = temp_audio.name

    try:

        tts = gTTS(
            text=text,
            lang=lang
        )

        tts.save(audio_path)

        return audio_path

    except Exception:

        if os.path.exists(audio_path):
            os.remove(audio_path)

        raise

# ----------------------------------
# Language Detection
# ----------------------------------

def detect_language(text):

    detected_lang = detect(text)

    # Map language codes unsupported by gTTS
    # or inconsistent with translation codes
    language_map = {
        "zh-cn": "zh-CN",
        "zh-tw": "zh-TW"
    }

    return language_map.get(
        detected_lang,
        detected_lang
    )

# ----------------------------------
# Process User Message
# ----------------------------------

def process_message(user_text):

    detected_lang = detect_language(user_text)

    st.info(
        f"Detected language: {detected_lang}"
    )

    # Translate user message to English
    english_text = translate_text(
        user_text,
        "auto",
        "en"
    )

    st.write(
        "**English translation:**",
        english_text
    )

    # Generate AI response
    ai_response = generate_response(
        english_text
    )

    # Translate response into user's language
    translated_response = translate_text(
        ai_response,
        "en",
        detected_lang
    )

    return detected_lang, translated_response

# ----------------------------------
# Application UI
# ----------------------------------

st.title("🌍 Multilingual AI Voice Assistant")

st.markdown("""
Speak or type in English, Tamil, or Hindi.

- 🎤 Whisper converts voice into text.
- 🌐 Language detection identifies the language.
- 🔄 Translation converts the message into English.
- 🤖 FLAN-T5 generates the AI response.
- 🔊 gTTS speaks the translated response.
""")

# ----------------------------------
# Session State
# ----------------------------------

if "history" not in st.session_state:
    st.session_state.history = []

# ----------------------------------
# Voice Input
# ----------------------------------

st.subheader(" Record Voice")

audio_bytes = audio_recorder(
    pause_threshold=2.0
)

if audio_bytes:

    audio_path = None
    mp3_path = None

    try:

        with tempfile.NamedTemporaryFile(
            delete=False,
            suffix=".wav"
        ) as temp_audio:

            temp_audio.write(audio_bytes)
            audio_path = temp_audio.name

        st.audio(
            audio_bytes,
            format="audio/wav"
        )

        # Whisper speech recognition
        user_text = speech_to_text(
            audio_path
        )

        if not user_text:
            st.warning("No speech detected.")

        else:

            st.success(
                f"Recognized text: {user_text}"
            )

            # Process message
            detected_lang, response = process_message(
                user_text
            )

            st.subheader("🤖 Assistant Response")
            st.write(response)

            # Save conversation
            st.session_state.history.append(
                (user_text, response)
            )

            # Generate voice output
            mp3_path = text_to_speech(
                response,
                detected_lang
            )

            with open(mp3_path, "rb") as audio_file:
                st.audio(
                    audio_file.read(),
                    format="audio/mp3"
                )

    except Exception as e:

        st.error(f"Error: {e}")

    finally:

        if audio_path and os.path.exists(audio_path):
            os.remove(audio_path)

        if mp3_path and os.path.exists(mp3_path):
            os.remove(mp3_path)

# ----------------------------------
# Text Chat
# ----------------------------------

st.subheader("💬 Text Chat")

user_input = st.text_input(
    "Enter your message"
)

if st.button("Send"):

    if user_input.strip():

        mp3_path = None

        try:

            detected_lang, response = process_message(
                user_input.strip()
            )

            st.subheader("🤖 Assistant Response")
            st.write(response)

            st.session_state.history.append(
                (user_input.strip(), response)
            )

            # Generate audio response
            mp3_path = text_to_speech(
                response,
                detected_lang
            )

            with open(mp3_path, "rb") as audio_file:
                st.audio(
                    audio_file.read(),
                    format="audio/mp3"
                )

        except Exception as e:

            st.error(f"Error: {e}")

        finally:

            if mp3_path and os.path.exists(mp3_path):
                os.remove(mp3_path)

# ----------------------------------
# Conversation History
# ----------------------------------

st.subheader("📜 Conversation History")

for question, answer in reversed(
    st.session_state.history
):

    st.markdown(f"**👤 User:** {question}")
    st.markdown(f"**🤖 Assistant:** {answer}")
    st.divider()